### Setup

In [ ]:
# HuggingFace ecosystem
%pip install transformers==4.40.2 numpy==1.26.4 datasets accelerate -U -q

# PyTorch
%pip install torch==2.1.2 --index-url https://download.pytorch.org/whl/cu121 -q

# Utilities
%pip install ipywidgets==7.8.5 pyarrow scikit-learn python-dotenv -U -q

In [ ]:
import os
from pathlib import Path

current_dir = Path.cwd()

if current_dir.name == "notebooks" and (
    Path.exists(current_dir.parent / Path("configs/fine_tune"))
    and Path.exists(current_dir.parent / Path("data"))
):
    os.chdir(current_dir.parent)
    print(f"Current directory: {Path.cwd().relative_to(Path.home())}")
elif current_dir.name == "dt133g-thesis-project":
    print(f"Current directory: {Path.cwd().relative_to(Path.home())}")
else:
    print("Ensure configs and data exists before running this notebook..")

### Imports and Constants

In [ ]:
import notebook_init  # Set visible GPUs
import torch
import pandas as pd
from tqdm.auto import tqdm
from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)

from utils import load_model, load_tokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

LABEL_ID = {
    "HUMAN_GENERATED": 0,
    "MACHINE_GENERATED": 1
}

LABEL_TEXT = {
    0: "HUMAN_GENERATED",
    1: "MACHINE_GENERATED"
}

pretrained_models = {
    "cb": "microsoft/codebert-base",
    "gc": "microsoft/graphcodebert-base",
    "ux": "microsoft/unixcoder-base",
    "ct": "Salesforce/codet5p-770",
    "ds": "deepseek-ai/deepseek-coder-1.3b-base",
}

In [ ]:
WINDOW = 512
MAX_NEW_TOKENS = 5

MODEL_NAME = pretrained_models["cb"]
DATASET_NAME = "codet_m4"

MODEL_PATH = f"data/models/{DATASET_NAME}/{MODEL_NAME}"

MODEL_TYPE = "encoder"
if "codet5" in MODEL_NAME:
    MODEL_TYPE = "seq2seq"
elif "deepseek" in MODEL_NAME:
    MODEL_TYPE = "causal"
print("MODEL_TYPE =", MODEL_TYPE)

SAVE_DIR = MODEL_PATH / Path("eval_outputs")
SAVE_DIR.mkdir(exist_ok=True)

### Predict with probability

In [ ]:
def predict_proba(model, tokenizer, sample):
    """
    Unified probability prediction for:
    - seq2seq models (CodeT5)
    - causal LMs (DeepSeek)

    Returns:
        {
            "pred_label": int,
            "probabilities": np.ndarray,
            "logits": np.ndarray
        }
    """

    model.to(DEVICE)
    model.eval()

    if MODEL_TYPE == "encoder":
        enc = tokenizer(
            sample["code"],
            truncation=True,
            padding="max_length",
            max_length=512,
            return_tensors="pt"
        )

        inputs = {k: v.to(DEVICE) for k, v in enc.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        logits = outputs.logits[0]
        probs = torch.softmax(logits, dim=-1)
        pred_label = probs.argmax().item()

        return {
            "pred_label": pred_label,
            "probabilities": probs.cpu().numpy(),
            "logits": logits.cpu().numpy()
        }

    elif MODEL_TYPE == "seq2seq":
        inputs = {
            "input_ids": torch.tensor(
                [sample["input_ids"]],
                dtype=torch.long,
                device=DEVICE
            ),
            "attention_mask": torch.tensor(
                [sample["attention_mask"]],
                dtype=torch.long,
                device=DEVICE
            ),
        }

        label_texts = [
            ("HUMAN_GENERATED", 0),
            ("MACHINE_GENERATED", 1)
        ]

        losses = []

        with torch.no_grad():
            for text, label_id in label_texts:

                label_ids = tokenizer(
                    text,
                    return_tensors="pt"
                ).input_ids.to(DEVICE)

                out = model(
                    **inputs,
                    labels=label_ids
                )

                losses.append(out.loss.item())
        # lower loss = better
        logits = -torch.tensor(losses)
        probs = torch.softmax(logits, dim=-1)
        pred_label = probs.argmax().item()

        return {
            "pred_label": pred_label,
            "probabilities": probs.cpu().numpy(),
            "logits": logits.cpu().numpy()
        }

    elif MODEL_TYPE == "causal":
        prompt = (
            "Classify code as human or machine.\n\n"
            + sample["raw_code"]
            + "\n\nAnswer:"
        )

        inputs = tokenizer(
            prompt,
            return_tensors="pt"
        ).to(DEVICE)

        with torch.no_grad():
            outputs = model(**inputs)

        next_token_logits = outputs.logits[0, -1, :]

        human_id = tokenizer.encode(
            "human",
            add_special_tokens=False
        )[0]

        machine_id = tokenizer.encode(
            "machine",
            add_special_tokens=False
        )[0]

        logits = torch.tensor([
            next_token_logits[human_id],
            next_token_logits[machine_id]
        ])
        probs = torch.softmax(logits, dim=-1)
        pred_label = probs.argmax().item()

        return {
            "pred_label": pred_label,
            "probabilities": probs.cpu().numpy(),
            "logits": logits.cpu().numpy()
        }

    else:
        raise ValueError(f"Unsupported model type: {MODEL_TYPE}")

### Tier 0 - Baseline  
Evaluation on unmutated test dataset

In [ ]:
DATASET_PATH = f"data/_06_generated_splits/{DATASET_NAME}/test.parquet"

In [ ]:
dataset = Dataset.from_parquet(
    DATASET_PATH,
    columns=["code", "label"]
)

print(dataset)

##### TODO - Turn into reusable functions:

In [ ]:
results = []

tokenizer = load_tokenizer(MODEL_PATH)
model = load_model(MODEL_PATH, tokenizer, DEVICE)

for sample in tqdm(dataset):
    code = sample["code"]
    label = int(sample["label"])

    pred_proba = predict_proba(model, tokenizer, sample)

    results.append({
        "label": label,
        "prediction": pred_proba["pred_label"],
        "prediction_text": LABEL_TEXT[pred_proba["pred_label"]],
        "code": code,
    })

df = pd.DataFrame(results)

print(df.head())

In [ ]:
labels = df["label"].tolist()
preds = df["prediction"].tolist()

acc = accuracy_score(labels, preds)
precision, recall, f1, _ = precision_recall_fscore_support(
    labels,
    preds,
    average="macro",
    zero_division=0,
)

print("Accuracy:", acc)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

print()
print(classification_report(labels, preds))

In [ ]:
cm = confusion_matrix(labels, preds)
print(cm)

In [ ]:
output_csv = SAVE_DIR / "predictions.csv"
output_json = SAVE_DIR / "predictions.json"


df.to_csv(output_csv, index=False)

df.to_json(
    output_json,
    orient="records",
    indent=2,
)

print("Saved:")
print(output_csv)
print(output_json)

### Tier 1 - Surface-Level Normalization
Evaluation on mutated test dataset

In [ ]:
TIER = "tier_1"
DATASET_PATH = f"data/transformations/{DATASET_NAME}/{TIER}/augmented_dataset.parquet"

In [ ]:
dataset = Dataset.from_parquet(
    DATASET_PATH,
    columns=["mutated_code", "label"]
)

print(dataset)

##### TODO - Add upcoming reusable functions and the rest of the tiers ...